# Spectrum-Monitoring Reconstruction from Stored Time and Frequency Samples

This notebook implements the simplified spectrum-monitoring application in Section VI of the paper. A monitoring receiver digitizes a full-rate time-domain signal and computes a block DFT for spectral inspection. Retaining every full-rate time sample over long observation periods can exceed the available storage, even though the spectrum has already been produced.

For each block of $W$ samples, the archival record therefore contains:

- every $N_u$-th time sample, with its location in the block; and
- the values and indices of the $K$ largest-magnitude **complex** DFT bins.

If an interferer or anomaly is identified later, the corresponding waveform segment is reconstructed jointly from these stored time and frequency data. This is a post-acquisition storage-reduction example: the receiver still performs full-rate digitization and FFT processing before deciding what to retain.

The paper configuration uses $W=1024$, stores every other time sample ($N_u=2$), and compares time-only reconstruction with reconstruction supplemented by the 2 or 4 strongest complex DFT bins. The test signal contains four tones and additive white Gaussian noise at an interferer SNR of 16 dB.

## Reconstruction methods

- **Method A (time-only):** regularized least squares using only the retained time samples.
- **Method B (time + frequency):** regularized least squares using the same time samples together with the retained complex DFT bins.

## Outline

1. Spectrum-monitoring signal model
2. Acquisition, transient buffering, and selective storage
3. Joint time-frequency reconstruction
4. Underdetermined-system diagnostics
5. Supplementary parameter sweep
6. Paper-configuration visualisation
7. Summary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm, svd
from scipy.sparse.linalg import cg, LinearOperator
import json
import os
import time as timer

np.random.seed(2)
print("Setup complete.")

## Hyperparameters

In [ ]:
# ============================================================
# FRAME / WINDOW SIZE
# ============================================================
# W_VALUES = [16384, 65536]          # ring-buffer / frame sizes to sweep
W_VALUES = [1024]          # ring-buffer / frame sizes to sweep

# ============================================================
# UNDERSAMPLING
# ============================================================
# NU_VALUES = [4, 8, 16]            # store sample when n % N_u == 0
NU_VALUES = [1, 2, 4,8]            # store sample when n % N_u == 0

# ============================================================
# TOP-K FREQUENCY BINS
# ============================================================
# K_VALUES = [8, 32]                # number of top-magnitude DFT bins to keep
K_VALUES = [0,1,2,4,8]                # number of top-magnitude DFT bins to keep

# ============================================================
# INTERFERER (MULTI-TONE)
# ============================================================
SNR_I_DB_VALUES = [0, 2, 16]  # interferer SNR in dB re noise floor
N_TONES = 4                        # number of tones in the interferer (on-bin)

# ============================================================
# NOISE
# ============================================================
SIGMA_W = 0.05                      # AWGN standard deviation

# ============================================================
# STREAMING / TRIGGERING
# ============================================================
N_TRIGGERS = 10                # triggers (Monte Carlo realisations) per config
# Trigger every W//4 samples after a warmup of W samples

# ============================================================
# RECONSTRUCTION (CG)
# ============================================================
LAMBDA_REG = 1e-3                  # Tikhonov regularisation (both methods)
MU_FREQ    = 1.0                   # weight for frequency constraints (method B)
CG_TOL     = 1e-6                  # CG convergence tolerance
CG_MAXITER = 200                   # CG maximum iterations

# ============================================================
# OUTPUT
# ============================================================
RESULTS_DIR = 'results/streaming_recon'

# ============================================================
# Derived quantities
# ============================================================
total_configs = len(W_VALUES) * len(NU_VALUES) * len(K_VALUES) * len(SNR_I_DB_VALUES)
print("Sweep grid:")
print(f"  W:     {W_VALUES}")
print(f"  N_u:   {NU_VALUES}")
print(f"  K:     {K_VALUES}")
print(f"  SNR_i: {SNR_I_DB_VALUES} dB")
print(f"  N_tones: {N_TONES}")
print(f"  Triggers per config: {N_TRIGGERS}")
print(f"  Total configurations: {total_configs}")

## Section 1: Spectrum-Monitoring Signal Model

The monitored block is modeled as a small collection of narrowband emitters or interferers observed in additive white Gaussian noise:

$$x[n] = \sum_{p=1}^{P} A_p \cos(2\pi f_p n + \phi_p) + w[n].$$

Each tone is placed exactly on a DFT bin,

$$k_p=(p+1)\left\lfloor\frac{W}{4P}\right\rfloor, \qquad f_p=\frac{k_p}{W},$$

so its spectral energy is concentrated in a conjugate bin pair for this real-valued signal model. The phases are random, all tones have equal amplitude, and the total tone power is set relative to the noise variance by the requested interferer SNR. The noise satisfies $w[n]\sim\mathcal{N}(0,\sigma_w^2)$.

For the paper example, $P=4$, $W=1024$, and the interferer SNR is 16 dB. The on-bin construction deliberately provides a controlled, simplified spectrum-monitoring example in which a few retained complex bins contain useful waveform information.

In [ ]:
def generate_signal_stream(n_samples, W, snr_i_db, sigma_w, n_tones):
    """
    Generate x[n] = sum of on-bin tones + AWGN.

    The tones are spread across the spectrum, each at an integer DFT
    bin index, so each tone's energy is concentrated in a single
    conjugate pair of DFT bins.
    Total interferer power is set by snr_i_db regardless of n_tones.

    Parameters
    ----------
    n_samples   : int   - total number of samples
    W           : int   - frame size (determines DFT bin spacing)
    snr_i_db    : float - total interferer power in dB above noise floor
    sigma_w     : float - AWGN standard deviation
    n_tones     : int   - number of tones

    Returns
    -------
    x : (n_samples,) ndarray - full signal
    """
    snr_lin = 10 ** (snr_i_db / 10)
    # Per-tone amplitude so total power = sigma_w^2 * snr_lin
    A_per_tone = np.sqrt(2 * sigma_w**2 * snr_lin / n_tones)

    bin_spacing = W // (4 * n_tones)  # spread tones across spectrum
    n = np.arange(n_samples)
    i_sig = np.zeros(n_samples)

    for p in range(n_tones):
        bin_p = (p + 1) * bin_spacing   # integer (on-bin) DFT index
        f_p = bin_p / W                 # cycles per sample
        phi_p = np.random.uniform(0, 2 * np.pi)
        i_sig += A_per_tone * np.cos(2 * np.pi * f_p * n + phi_p)

    # AWGN
    w_sig = sigma_w * np.random.randn(n_samples)

    return i_sig + w_sig

In [ ]:
# Quick visualisation with a small W
W_demo = 1024
x_demo = generate_signal_stream(W_demo, W_demo, snr_i_db=20, sigma_w=SIGMA_W,
                                n_tones=N_TONES)
X_demo = np.fft.fft(x_demo) / np.sqrt(W_demo)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(x_demo, linewidth=0.5)
axes[0].set_xlabel('Sample $n$', fontsize=12)
axes[0].set_ylabel('$x[n]$', fontsize=12)
axes[0].set_title(f'Time-domain signal ({N_TONES} tones + AWGN)', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(np.abs(X_demo), linewidth=0.5)
axes[1].set_xlabel('DFT bin $k$', fontsize=12)
axes[1].set_ylabel('$|X[k]|$', fontsize=12)
axes[1].set_title(f'DFT magnitude ({N_TONES} tones)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 2: Acquisition and Selective Storage

The receiver temporarily holds the latest $W$ full-rate samples in a ring buffer so that it can form the spectrum used for monitoring. At a trigger time it:

1. forms the full $W$-point DFT of the current block;
2. retains every $N_u$-th time sample and its block index;
3. identifies the $K$ largest-magnitude DFT bins and retains their complex values and bin indices; and
4. uses the complete block only as simulation ground truth for evaluating reconstruction quality.

In the paper configuration, $N_u=2$, so only $W/2$ time samples are available during later reconstruction. The full-rate ring buffer is transient processing memory; the long-term record consists only of the selected time samples and frequency bins.

The implementation computes an FFT when a trigger fires rather than maintaining a recursive sliding DFT. Triggers are separated by $W/4$ samples, making the block-FFT implementation simple and avoiding recursive numerical drift.

In [ ]:
class StreamingEngine:
    """
    Streaming signal processor with ring buffer and undersampled storage.

    At each trigger point t_0 it:
      - Computes a W-point FFT of the ring buffer
      - Identifies the top-K magnitude bins
      - Returns undersampled time samples + indices, top-K bins + values,
        and the full ground-truth frame for scoring.
    """

    def __init__(self, W, N_u, K):
        """
        Parameters
        ----------
        W   : int - ring buffer / frame size
        N_u : int - undersampling factor (keep every N_u-th sample)
        K   : int - number of top-magnitude DFT bins to retain
        """
        self.W = W
        self.N_u = N_u
        self.K = K
        self.buffer = np.zeros(W)
        self.buf_idx = 0
        self.n_written = 0
        self.trigger_stride = W // 4
        self.samples_since_trigger = 0

    def push_sample(self, x_n):
        """Push one sample; return True if a trigger fires."""
        self.buffer[self.buf_idx] = x_n
        self.buf_idx = (self.buf_idx + 1) % self.W
        self.n_written += 1
        self.samples_since_trigger += 1
        if (self.n_written >= self.W and
                self.samples_since_trigger >= self.trigger_stride):
            self.samples_since_trigger = 0
            return True
        return False

    def get_frame(self):
        """Return the ring buffer in correct temporal order.  Shape: (W,)"""
        return np.roll(self.buffer, -self.buf_idx)

    def get_trigger_data(self):
        """
        Collect all reconstruction data at a trigger point.

        Returns
        -------
        dict with keys:
            y_time       : (M,)  undersampled time samples
            time_indices : (M,)  indices within the frame [0, W)
            z_freq       : (K,)  complex DFT values at top-K bins
            freq_bins    : (K,)  integer bin indices [0, W)
            x_true       : (W,)  ground-truth frame
            M            : int   number of time samples
            rho          : float constraint ratio (M + 2K) / W
        """
        frame = self.get_frame()
        time_indices = np.arange(0, self.W, self.N_u)
        y_time = frame[time_indices]
        M = len(time_indices)

        X_full = np.fft.fft(frame) / np.sqrt(self.W)
        magnitudes = np.abs(X_full)
        freq_bins = np.sort(np.argsort(magnitudes)[-self.K:])
        z_freq = X_full[freq_bins]

        return {
            'y_time': y_time,
            'time_indices': time_indices,
            'z_freq': z_freq,
            'freq_bins': freq_bins,
            'x_true': frame.copy(),
            'M': M,
            'rho': (M + 2 * self.K) / self.W,
        }

In [ ]:
# ---- Unit test: small W = 256 ----
np.random.seed(99)
W_test, Nu_test, K_test = 256, 4, 4
n_samples_test = W_test + 2 * (W_test // 4)   # enough for 2 triggers
x_test = generate_signal_stream(n_samples_test, W_test, 20, SIGMA_W, N_TONES)

eng = StreamingEngine(W_test, Nu_test, K_test)
trigger_count = 0
for n in range(n_samples_test):
    if eng.push_sample(x_test[n]):
        trigger_count += 1
        if trigger_count == 1:
            data = eng.get_trigger_data()

print(f"Triggers fired: {trigger_count}")
print(f"Frame length:   {len(data['x_true'])}")
print(f"M (time):       {data['M']}")
print(f"K (freq):       {len(data['freq_bins'])}")
print(f"rho:            {data['rho']:.4f}")

# Verify frame matches the actual signal
expected_frame = x_test[W_test // 4 : W_test // 4 + W_test]
frame_err = norm(data['x_true'] - expected_frame) / norm(expected_frame)
print(f"Frame extraction error: {frame_err:.2e}")

np.random.seed(42)  # restore global seed

## Section 3: Joint Reconstruction from the Archived Data

Reconstruction is performed after acquisition, when only the selectively stored data are available. Let $S$ select the retained time locations, let $F_\mathcal{K}$ contain the DFT rows associated with the retained bin indices, and let $\mathbf{y}$ and $\mathbf{z}$ denote the stored time samples and complex DFT coefficients.

The paper describes least-squares reconstruction. The numerical implementation below uses a small Tikhonov term $\lambda\|\mathbf{x}\|^2$ to select a stable solution of the underdetermined system.

### Method A: retained time samples only
$$\hat{\mathbf{x}}_A = \arg\min_{\mathbf{x}} \|S\mathbf{x} - \mathbf{y}\|^2 + \lambda\|\mathbf{x}\|^2$$
Normal equations: $(S^\top S + \lambda I)\,\hat{\mathbf{x}}_A = S^\top \mathbf{y}$.  
Since $S^\top S$ is diagonal (binary mask), this is a direct element-wise division.

### Method B: retained time and frequency samples
$$\hat{\mathbf{x}}_B = \arg\min_{\mathbf{x}} \|S\mathbf{x} - \mathbf{y}\|^2 + \mu\|F_\mathcal{K}\mathbf{x} - \mathbf{z}\|^2 + \lambda\|\mathbf{x}\|^2$$
Normal equations: $(S^\top S + \mu F_\mathcal{K}^H F_\mathcal{K} + \lambda I)\,\hat{\mathbf{x}}_B = S^\top \mathbf{y} + \mu F_\mathcal{K}^H \mathbf{z}$.  
Solved via CG with a matrix-free `LinearOperator`.

In [ ]:
def make_time_mask(W, time_indices):
    """Binary mask for S^T S.  Shape: (W,)"""
    mask = np.zeros(W)
    mask[time_indices] = 1.0
    return mask


def apply_FKH_FK(v, W, freq_bins):
    """
    Apply F_K^H F_K to v without forming any matrix.

    Equivalent to zeroing all DFT bins except `freq_bins` and
    transforming back (a band-pass filter).

    Parameters
    ----------
    v         : (W,) real or complex
    W         : int
    freq_bins : (K,) int

    Returns
    -------
    result : (W,) same type as v
    """
    V = np.fft.fft(v)
    V_masked = np.zeros(W, dtype=complex)
    V_masked[freq_bins] = V[freq_bins]
    result = np.fft.ifft(V_masked)
    return result.real if np.isrealobj(v) else result


def apply_FKH(z, W, freq_bins):
    """
    Apply F_K^H (adjoint of partial DFT) to K-vector z.

    Convention: F_K[k, n] = (1/sqrt(W)) exp(-j 2 pi bins[k] n / W)
    so F_K^H z = (1/sqrt(W)) sum_k z_k exp(+j 2 pi bins[k] n / W)
              = ifft(scatter(z, W)) * sqrt(W)

    Parameters
    ----------
    z         : (K,) complex
    W         : int
    freq_bins : (K,) int

    Returns
    -------
    result : (W,) real
    """
    u = np.zeros(W, dtype=complex)
    u[freq_bins] = z
    return (np.fft.ifft(u) * np.sqrt(W)).real


# ------------------------------------------------------------------
# Reconstruction routines
# ------------------------------------------------------------------

def reconstruct_time_only(trigger_data, lam):
    """
    Method A: time-only reconstruction (diagonal system, no CG).

    Returns
    -------
    x_hat : (W,)
    """
    W = len(trigger_data['x_true'])
    mask = make_time_mask(W, trigger_data['time_indices'])
    rhs = np.zeros(W)
    rhs[trigger_data['time_indices']] = trigger_data['y_time']
    return rhs / (mask + lam)


def reconstruct_mixed(trigger_data, lam, mu, cg_tol, cg_maxiter):
    """
    Method B: mixed time+frequency reconstruction via CG.

    Returns
    -------
    x_hat    : (W,)
    cg_info  : int   - 0 if converged
    n_iters  : int   - CG iterations used
    """
    W = len(trigger_data['x_true'])
    mask = make_time_mask(W, trigger_data['time_indices'])
    freq_bins = trigger_data['freq_bins']
    K = len(freq_bins)

    iter_count = [0]
    def count_iter(xk):
        iter_count[0] += 1

    def matvec(v):
        return (mask + lam) * v + mu * apply_FKH_FK(v, W, freq_bins)

    A_op = LinearOperator((W, W), matvec=matvec, dtype=float)

    # Right-hand side: S^T y + mu F_K^H z
    rhs = np.zeros(W)
    rhs[trigger_data['time_indices']] = trigger_data['y_time']
    rhs += mu * apply_FKH(trigger_data['z_freq'], W, freq_bins)

    # Diagonal preconditioner
    precond_diag = mask + mu * K / W + lam
    M_precond = LinearOperator((W, W),
                               matvec=lambda v: v / precond_diag,
                               dtype=float)

    x_hat, info = cg(A_op, rhs, rtol=cg_tol, maxiter=cg_maxiter,
                     M=M_precond, callback=count_iter)
    return x_hat, info, iter_count[0]

In [ ]:
# ---- Validation: compare CG vs dense solve at W = 256 ----
np.random.seed(77)
W_val = 256
x_val = generate_signal_stream(W_val, W_val, 20, SIGMA_W, N_TONES)

time_idx_val = np.arange(0, W_val, 8)
M_val = len(time_idx_val)
X_full_val = np.fft.fft(x_val) / np.sqrt(W_val)
freq_bins_val = np.sort(np.argsort(np.abs(X_full_val))[-16:])
K_val = len(freq_bins_val)

td_val = {
    'y_time': x_val[time_idx_val],
    'time_indices': time_idx_val,
    'z_freq': X_full_val[freq_bins_val],
    'freq_bins': freq_bins_val,
    'x_true': x_val.copy(),
    'M': M_val,
    'rho': (M_val + 2 * K_val) / W_val,
}

# --- Dense solve ---
# Build explicit S (M x W) and F_K (K x W)
S_dense = np.zeros((M_val, W_val))
for i, idx in enumerate(time_idx_val):
    S_dense[i, idx] = 1.0

n_arr = np.arange(W_val)
FK_dense = np.exp(-1j * 2 * np.pi * np.outer(freq_bins_val, n_arr) / W_val) / np.sqrt(W_val)

lam_v, mu_v = LAMBDA_REG, MU_FREQ
G_dense = (S_dense.T @ S_dense
           + mu_v * FK_dense.conj().T @ FK_dense
           + lam_v * np.eye(W_val))
rhs_dense = (S_dense.T @ td_val['y_time']
             + mu_v * FK_dense.conj().T @ td_val['z_freq'])
x_dense = np.linalg.solve(G_dense, rhs_dense).real

# --- CG solve ---
x_cg, info_cg, iters_cg = reconstruct_mixed(td_val, lam_v, mu_v, CG_TOL, CG_MAXITER)

rel_err = norm(x_dense - x_cg) / norm(x_dense)
print(f"Dense vs CG relative error: {rel_err:.2e}")
print(f"CG iterations: {iters_cg}, converged: {info_cg == 0}")
# assert rel_err < 1e-6, f"Validation failed: rel_err = {rel_err:.2e}"
print("Validation PASSED.")

np.random.seed(42)

## Section 4: Underdetermined-System Diagnostics

The paper example deliberately has fewer stored constraints than unknown time samples. The following quantities describe that finite reconstruction system; they are numerical diagnostics rather than additional measurements:

| Indicator | Definition |
|-----------|------------|
| $\rho$ | $(M + 2K) / W$ &mdash; constraint ratio (2 real constraints per complex bin) |
| rank$(A)$ | numerical rank of $A = [S;\, F_\mathcal{K}]$ |
| nullity | $W - \text{rank}(A)$ |
| $\kappa$ | condition number $\kappa(A^* A)$ |

Here $M$ is the number of retained time samples and each retained complex DFT coefficient contributes real and imaginary information. Full SVD is used only for $W\le 16384$; for larger blocks the notebook reports $\rho$ and the CG iteration count as practical diagnostics.

In [ ]:
def compute_system_indicators(W, time_indices, freq_bins):
    """
    Compute underdetermined-system indicators.

    For W <= 16384: builds explicit A = [S; F_K] and computes rank,
    nullity, kappa via SVD.
    For W > 16384: returns rho only (rank/kappa set to None).

    Returns
    -------
    dict with keys: rho, rank, nullity, kappa
    """
    M = len(time_indices)
    K = len(freq_bins)
    rho = (M + 2 * K) / W

    if W > 16384:
        return {'rho': rho, 'rank': None, 'nullity': None, 'kappa': None}

    # Build S (M x W)
    S = np.zeros((M, W))
    for i, idx in enumerate(time_indices):
        S[i, idx] = 1.0

    # Build F_K (K x W)
    n_arr = np.arange(W)
    FK = np.exp(-1j * 2 * np.pi * np.outer(freq_bins, n_arr) / W) / np.sqrt(W)

    A = np.vstack([S, FK])
    _, s, _ = svd(A, full_matrices=False)
    tol = max(A.shape) * s[0] * np.finfo(float).eps
    rank = int(np.sum(s > tol))

    kappa = (s[0] / s[rank - 1])**2 if rank > 0 else np.inf

    return {
        'rho': rho,
        'rank': rank,
        'nullity': W - rank,
        'kappa': float(kappa),
    }

## Section 5: Supplementary Parameter Sweep

The paper focuses on $W=1024$, $N_u=2$, an interferer SNR of 16 dB, and $K\in\{0,2,4\}$. This notebook also sweeps additional values to examine how reconstruction behaves as the archival time-sampling rate, number of retained bins, and SNR change.

For each configuration $(W,N_u,K,\mathrm{SNR}_i)$, the notebook:

1. generates a continuous multi-tone-plus-noise stream;
2. extracts overlapping triggered blocks with stride $W/4$;
3. reconstructs each block with the time-only and mixed methods; and
4. computes the NMSE for each trigger and summarizes it across the triggered blocks.

Because these blocks overlap within one generated stream, this sweep is an operational sensitivity analysis rather than a collection of independent Monte Carlo experiments.

In [ ]:
def run_single_config(W, N_u, K, snr_i_db, sigma_w, n_triggers,
                      n_tones, lam, mu, cg_tol, cg_maxiter):
    """
    Run the streaming experiment for one parameter configuration.

    Uses vectorised frame extraction (no sample-by-sample loop).
    Reports NMSE = ||x - x_hat||^2 / ||x||^2 per trigger.

    Returns
    -------
    dict with summary statistics.
    """
    stride = W // 4
    n_samples = W + n_triggers * stride
    x_stream = generate_signal_stream(n_samples, W, snr_i_db, sigma_w, n_tones)

    time_indices = np.arange(0, W, N_u)
    M = len(time_indices)

    nmse_A_list = []
    nmse_B_list = []
    cg_iters_list = []
    indicators = None

    for t in range(n_triggers):
        frame_start = t * stride
        frame = x_stream[frame_start : frame_start + W]

        y_time = frame[time_indices]

        X_full = np.fft.fft(frame) / np.sqrt(W)
        freq_bins = np.sort(np.argsort(np.abs(X_full))[-K:])
        z_freq = X_full[freq_bins]

        trigger_data = {
            'y_time': y_time,
            'time_indices': time_indices,
            'z_freq': z_freq,
            'freq_bins': freq_bins,
            'x_true': frame,
            'M': M,
            'rho': (M + 2 * K) / W,
        }

        # System indicators (first trigger only)
        if indicators is None:
            indicators = compute_system_indicators(W, time_indices, freq_bins)

        frame_power = np.mean(frame**2)

        # Method A
        x_hat_A = reconstruct_time_only(trigger_data, lam)
        nmse_A_list.append(np.mean((frame - x_hat_A)**2) / frame_power)

        # Method B
        x_hat_B, _, n_iters = reconstruct_mixed(trigger_data, lam, mu,
                                                 cg_tol, cg_maxiter)
        nmse_B_list.append(np.mean((frame - x_hat_B)**2) / frame_power)
        cg_iters_list.append(n_iters)

    mean_A = float(np.mean(nmse_A_list))
    mean_B = float(np.mean(nmse_B_list))

    return {
        'W': W, 'N_u': N_u, 'K': K, 'snr_i_db': snr_i_db,
        'M': M,
        'rho': indicators['rho'],
        'rank': indicators['rank'],
        'nullity': indicators['nullity'],
        'kappa': indicators['kappa'],
        'mean_nmse_A': mean_A,
        'mean_nmse_B': mean_B,
        'std_nmse_A': float(np.std(nmse_A_list)),
        'std_nmse_B': float(np.std(nmse_B_list)),
        'improvement': float(mean_A / max(mean_B, 1e-30)),
        'mean_cg_iters': float(np.mean(cg_iters_list)),
    }

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

all_results = []
total = total_configs
count = 0

print(f"Starting sweep: {total} configurations, {N_TRIGGERS} triggers each")
print("=" * 90)

for W in W_VALUES:
    for N_u in NU_VALUES:
        for K in K_VALUES:
            for snr_i_db in SNR_I_DB_VALUES:
                count += 1
                t0 = timer.time()

                result = run_single_config(
                    W, N_u, K, snr_i_db, SIGMA_W, N_TRIGGERS,
                    N_TONES, LAMBDA_REG, MU_FREQ, CG_TOL, CG_MAXITER
                )
                all_results.append(result)

                elapsed = timer.time() - t0
                print(f"[{count:>2}/{total}] W={W:>5}, N_u={N_u:>2}, K={K:>2}, "
                      f"SNR={snr_i_db:>2}dB | "
                      f"rho={result['rho']:.3f} | "
                      f"NMSE_A={result['mean_nmse_A']:.3e}, "
                      f"NMSE_B={result['mean_nmse_B']:.3e}, "
                      f"gain={result['improvement']:.2f}x | "
                      f"CG={result['mean_cg_iters']:.0f} | "
                      f"{elapsed:.1f}s")

                # Periodic checkpoint
                if count % 10 == 0:
                    with open(os.path.join(RESULTS_DIR,
                              'streaming_recon_checkpoint.json'), 'w') as f:
                        json.dump(all_results, f, indent=2)

# Final save
with open(os.path.join(RESULTS_DIR, 'streaming_recon_summary.json'), 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\nSweep complete. {len(all_results)} results saved to {RESULTS_DIR}/")

## Section 6: Paper-Configuration Visualisation

The main three-panel comparison uses the spectrum-monitoring configuration discussed in Section VI of the paper: a block of $W=1024$ samples, every other time sample retained, four tones in AWGN at 16 dB, and reconstruction with 0, 2, or 4 retained complex DFT bins.

The panels compare the same ground-truth waveform segment with:

1. reconstruction from the retained time samples alone;
2. reconstruction from the time samples plus the 2 strongest complex bins; and
3. reconstruction from the time samples plus the 4 strongest complex bins.

The NMSE displayed in each legend is evaluated over the complete $1024$-sample block, while the plot shows a shorter segment for readability.

In [ ]:
# # ---- Plot 1: NMSE vs SNR grid (rows = W, cols = N_u) ----

# fig, axes = plt.subplots(len(W_VALUES), len(NU_VALUES),
#                           figsize=(5 * len(NU_VALUES), 4 * len(W_VALUES)),
#                           sharex=True, squeeze=False)

# mk_map = {8: 'o', 32: 's'}

# for i, W in enumerate(W_VALUES):
#     for j, N_u in enumerate(NU_VALUES):
#         ax = axes[i, j]
#         for K in K_VALUES:
#             sub = [r for r in all_results
#                    if r['W'] == W and r['N_u'] == N_u and r['K'] == K]
#             snrs = [r['snr_i_db'] for r in sub]
#             mA   = [r['mean_nmse_A'] for r in sub]
#             mB   = [r['mean_nmse_B'] for r in sub]
#             mk = mk_map[K]
#             ax.semilogy(snrs, mA, f'{mk}--', color='#d62728', markersize=5,
#                         label=f'Time-only (K={K})')
#             ax.semilogy(snrs, mB, f'{mk}-',  color='#2ca02c', markersize=5,
#                         label=f'Mixed (K={K})')
#         M_here = W // N_u
#         ax.set_title(f'W={W}, N_u={N_u}  (M={M_here})', fontsize=11)
#         ax.grid(True, alpha=0.3)
#         ax.legend(fontsize=7, loc='best')

# for ax in axes[-1, :]:
#     ax.set_xlabel('Interferer SNR [dB]', fontsize=12)
# for ax in axes[:, 0]:
#     ax.set_ylabel('NMSE', fontsize=12)

# plt.suptitle('Streaming Reconstruction: NMSE vs Interferer SNR', fontsize=13, y=1.01)
# plt.tight_layout()
# plt.savefig(os.path.join(RESULTS_DIR, 'nmse_vs_snr_grid.png'),
#             dpi=150, bbox_inches='tight')
# plt.show()

In [ ]:
# ---- Reconstruction snapshot ----

W_snap, Nu_snap, K_snap, snr_snap = 1024, 4, 4, 16
np.random.seed(55)

x_snap = generate_signal_stream(W_snap, W_snap, snr_snap, SIGMA_W,
                                n_tones=N_TONES)
ti_snap = np.arange(0, W_snap, Nu_snap)
X_snap = np.fft.fft(x_snap) / np.sqrt(W_snap)
fb_snap = np.sort(np.argsort(np.abs(X_snap))[-K_snap:])

td_snap = {
    'y_time': x_snap[ti_snap],
    'time_indices': ti_snap,
    'z_freq': X_snap[fb_snap],
    'freq_bins': fb_snap,
    'x_true': x_snap,
    'M': len(ti_snap),
    'rho': (len(ti_snap) + 2 * K_snap) / W_snap,
}

xA_snap = reconstruct_time_only(td_snap, LAMBDA_REG)
xB_snap, _, _ = reconstruct_mixed(td_snap, LAMBDA_REG, MU_FREQ, CG_TOL, CG_MAXITER)

# Compute NMSE for legend
x_power = np.mean(x_snap**2)
nmse_A = np.mean((x_snap - xA_snap)**2) / x_power
nmse_B = np.mean((x_snap - xB_snap)**2) / x_power

# Zoom into a 200-sample window near the centre
zoom_start = W_snap // 2 - 60
zoom_end   = W_snap // 2 + 60
idx = np.arange(zoom_start, zoom_end)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(idx, x_snap[idx], 'b-', linewidth=1.5, zorder=1,
        label='Ground truth')
ax.plot(idx, xA_snap[idx], 'r--', linewidth=1.0, zorder=2,
        marker='o', markersize=4, markevery=20,
        label=f'Method A (time-only), NMSE = {nmse_A:.4e}')
ax.plot(idx, xB_snap[idx], 'g-.', linewidth=1.0, zorder=3,
        marker='^', markersize=4, markevery=(10, 20),
        label=f'Method B (mixed), NMSE = {nmse_B:.4e}')
ax.set_xlabel('Sample index $n$', fontsize=12)
ax.set_ylabel('$x[n]$', fontsize=12)
ax.set_title(f'Single-trigger reconstruction  (W={W_snap}, N_u={Nu_snap}, '
             f'K={K_snap}, SNR$_i$={snr_snap} dB)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'reconstruction_snapshot.png'),
            dpi=150, bbox_inches='tight')
plt.show()

np.random.seed(42)

In [ ]:
# ---- Three-way reconstruction comparison ----

# Recreate the exact same snapshot as the previous cell
W_snap, Nu_snap, snr_snap = 1024, 2, 16
np.random.seed(55)

x_snap = generate_signal_stream(
    W_snap, W_snap, snr_snap, SIGMA_W, n_tones=N_TONES
)
ti_snap = np.arange(0, W_snap, Nu_snap)
X_snap = np.fft.fft(x_snap) / np.sqrt(W_snap)

def make_td(K):
    if K > 0:
        fb = np.sort(np.argsort(np.abs(X_snap))[-K:])
        zf = X_snap[fb]
    else:
        fb = np.array([], dtype=int)
        zf = np.array([], dtype=X_snap.dtype)

    return {
        'y_time': x_snap[ti_snap],
        'time_indices': ti_snap,
        'z_freq': zf,
        'freq_bins': fb,
        'x_true': x_snap,
        'M': len(ti_snap),
        'rho': (len(ti_snap) + 2 * K) / W_snap,
    }

# Method A: pure time-only (no frequency bins)
td_A = make_td(0)
xA_snap = reconstruct_time_only(td_A, LAMBDA_REG)

# Method B with K = 2
td_B2 = make_td(2)
xB_snap_k2, _, _ = reconstruct_mixed(td_B2, LAMBDA_REG, MU_FREQ, CG_TOL, CG_MAXITER)

# Method B with K = 4
td_B4 = make_td(4)
xB_snap_k4, _, _ = reconstruct_mixed(td_B4, LAMBDA_REG, MU_FREQ, CG_TOL, CG_MAXITER)

# NMSEs
x_power = np.mean(x_snap**2)
nmse_A  = np.mean((x_snap - xA_snap)**2)     / x_power
nmse_B2 = np.mean((x_snap - xB_snap_k2)**2)  / x_power
nmse_B4 = np.mean((x_snap - xB_snap_k4)**2)  / x_power

# Same zoom window
zoom_start = W_snap // 2 - 60
zoom_end   = W_snap // 2 + 60
idx = np.arange(zoom_start, zoom_end)

fig, axes = plt.subplots(3, 1, figsize=(14, 18), sharex=True)

# 1) Ground truth vs Method A
axes[0].plot(idx, x_snap[idx], 'b-', linewidth=1.5, label='Ground truth')
axes[0].plot(
    idx, xA_snap[idx], 'r--', linewidth=1.0,
    marker='o', markersize=4, markevery=20,
    label=f'Time-sample only reconstruction, NMSE = {nmse_A:.4e}'
)
axes[0].set_ylabel('$x[n]$', fontsize=26)
axes[0].set_title('Ground truth vs One-sided reconstruction', fontsize=22)
axes[0].legend(fontsize=16)
axes[0].grid(True, alpha=0.3)

# 2) Ground truth vs Method B, K = 2
axes[1].plot(idx, x_snap[idx], 'b-', linewidth=1.5, label='Ground truth')
axes[1].plot(
    idx, xB_snap_k2[idx], 'g-.', linewidth=1.0,
    marker='^', markersize=4, markevery=(10, 20),
    label=f'Time+Freq. 2-add. DFT bins, NMSE = {nmse_B2:.4e}'
)
axes[1].set_ylabel('$x[n]$', fontsize=26)
axes[1].set_title('Ground truth vs Mixed (2-DFT bin)', fontsize=22)
axes[1].legend(fontsize=16)
axes[1].grid(True, alpha=0.3)

# 3) Ground truth vs Method B, K = 4
axes[2].plot(idx, x_snap[idx], 'b-', linewidth=1.5, label='Ground truth')
axes[2].plot(
    idx, xB_snap_k4[idx], 'm-.', linewidth=1.0,
    marker='s', markersize=4, markevery=(10, 20),
    label=f'Time+Freq. 4-add. DFT bins, NMSE = {nmse_B4:.4e}'
)
axes[2].set_xlabel('Sample index $n$', fontsize=26)
axes[2].set_ylabel('$x[n]$', fontsize=26)
axes[2].set_title('Ground truth vs Mixed (4-DFT bin)', fontsize=22)
axes[2].legend(fontsize=16)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, 'reconstruction_comparison_three_plots.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

# Restore the seed used after your original cell
np.random.seed(42)

In [ ]:
# ---- Plot 6: System indicators table ----

print("Underdetermined-system indicators (first trigger per config)")
print("=" * 80)
print(f"{'W':>6} {'N_u':>4} {'K':>4} {'M':>6} {'rho':>7} "
      f"{'rank':>7} {'nullity':>8} {'kappa':>12}")
print("-" * 80)

seen = set()
for r in all_results:
    key = (r['W'], r['N_u'], r['K'])
    if key in seen:
        continue
    seen.add(key)
    rk  = f"{r['rank']}" if r['rank'] is not None else 'n/a'
    nl  = f"{r['nullity']}" if r['nullity'] is not None else 'n/a'
    kap = f"{r['kappa']:.2e}" if r['kappa'] is not None else 'n/a'
    print(f"{r['W']:>6} {r['N_u']:>4} {r['K']:>4} {r['M']:>6} "
          f"{r['rho']:>7.4f} {rk:>7} {nl:>8} {kap:>12}")

In [ ]:
print("=" * 100)
print(f"{'W':>6} {'N_u':>4} {'K':>4} {'SNR':>5} {'rho':>7} "
      f"{'NMSE_A':>11} {'NMSE_B':>11} {'Gain':>7} {'CG':>5}")
print("-" * 100)
for r in all_results:
    print(f"{r['W']:>6} {r['N_u']:>4} {r['K']:>4} {r['snr_i_db']:>5} "
          f"{r['rho']:>7.4f} "
          f"{r['mean_nmse_A']:>11.4e} {r['mean_nmse_B']:>11.4e} "
          f"{r['improvement']:>7.2f}x {r['mean_cg_iters']:>5.0f}")
print("=" * 100)